# Numerical Methods Week, Notebook 2: Interpolation

**Interpolation** is the problem of building a function that passes *exactly* through a
given set of points, so you can evaluate it in between.

The central fact is a uniqueness theorem, and it is worth stating carefully because it
shapes everything that follows:

> Given $n+1$ points with **distinct** $x$-values, there is **exactly one** polynomial of
> degree $\le n$ passing through all of them.

Exactly one. So Newton's form and Lagrange's form, which look nothing alike, are two
different *recipes for writing down the same polynomial*. They differ in cost, in
numerical behaviour, and in what happens when you add a point; not in the answer.

Three exercises:

1. **Newton's interpolating polynomial**: built from the divided differences of
   notebook 1, cheap to extend.
2. **Taylor approximation vs the real function**: the other way to approximate, using
   derivatives at *one* point instead of values at many, and how differently its error
   behaves.
3. **Lagrange interpolating polynomials**: the same polynomial, written to make the
   theory transparent.

Prerequisite: the **estimation** notebook, especially Exercise 1. Run the setup cell
first.

In [ ]:
# Setup: run this first.
# Import numpy as np and matplotlib.pyplot as plt.

---
# Exercise 1: Newton's interpolating polynomial

## The form

Newton writes the interpolating polynomial in this nested basis:

$$p(x) = c_0 + c_1(x - x_0) + c_2(x - x_0)(x - x_1) + \dots + c_n \prod_{j=0}^{n-1}(x - x_j)$$

and the miracle is that the coefficients are exactly the divided differences you already
computed:

$$c_k = f[x_0, x_1, \dots, x_k]$$

Check the first two by hand and you will believe the rest: at $x = x_0$ every term after
the first vanishes, so $c_0 = f(x_0)$. At $x = x_1$ only the first two survive, giving
$c_1 = (f(x_1) - f(x_0))/(x_1 - x_0)$: the first divided difference.

## Why this form is worth knowing

**Adding a data point costs one coefficient.** $c_0 \dots c_n$ do not change when you
append $x_{n+1}$; you just compute $c_{n+1}$ and add one term. No other form has this
property, and it is why Newton's form is used for adaptive schemes that keep refining
until the answer stops moving.

## Evaluating it: use Horner, not powers

Written naively, evaluation costs $O(n^2)$ multiplications. Nested (Horner) form costs
$O(n)$ and is more accurate:

$$p(x) = c_0 + (x - x_0)\Big(c_1 + (x - x_1)\big(c_2 + \dots\big)\Big)$$

which you compute by starting from the last coefficient and working backwards:

```
result = c[n]
for k in n-1, n-2, ..., 0:
    result = c[k] + (x - x[k]) * result
```

In [ ]:
# Step 1.1: Bring the coefficients across.
# Copy your  divided_difference_table(x, y)  and  newton_coefficients(x, y)  from the
# estimation notebook into this cell (or import them, if you saved them to a .py file,
# which is exactly the situation section 1 of the ecosystem notebook was about).
# Re-test on f(x) = x**3 - 2*x + 1 at nodes [-1, 0, 1, 2] to be sure they still work.

In [ ]:
# Step 1.2: Evaluate, with Horner.
# Write  newton_eval(coeffs, nodes, x)  implementing the nested form above.
# It must work when x is a numpy ARRAY as well as a single float: write it with
# numpy operations and it will, for free. Check that it does.

In [ ]:
# Step 1.3: The test that matters: does it interpolate?
# An interpolating polynomial must reproduce the data exactly.
# Evaluate your polynomial AT THE NODES and check it returns the original y values,
# using np.allclose. Do this for the cubic above.
# If this test fails nothing else in this exercise is meaningful, so stop and fix it.

In [ ]:
# Step 1.4: A second test: recover a known polynomial.
# f(x) = x**3 - 2*x + 1 has degree 3, so 4 nodes must reproduce it EVERYWHERE,
# not just at the nodes.
# Evaluate your interpolant on 200 points across [-2, 3] and compare with f itself.
# Print the maximum absolute difference. It should be at the level of round-off.

In [ ]:
# Step 1.5: Interpolate a function that is NOT a polynomial.
# Take f = np.sin on [0, pi] with 4 equally spaced nodes.
# Plot, on one figure: f on a fine grid, your interpolant on the same grid, and the
# nodes as scattered points. Label everything and add a legend.
# On a SECOND figure, plot the error f(x) - p(x). Where is it zero, and where is it
# largest? Explain the pattern in a comment.

In [ ]:
# Step 1.6: Refinement, and the incremental property.
# Repeat 1.5 for n = 2, 4, 8 and 16 equally spaced nodes, recording the maximum error
# on [0, pi] each time. Print the four numbers. How fast is the error falling?
#
# Then demonstrate the incremental property: build the coefficients for the first
# 4 nodes, then for those same 4 PLUS a fifth, and confirm the first four coefficients
# are unchanged.

In [ ]:
# Step 1.7: Runge's phenomenon. (Do not skip this one.)
# More nodes is not automatically better. Interpolate
#     f(x) = 1 / (1 + 25 * x**2)     on [-1, 1]
# with 6, 11 and 16 EQUALLY SPACED nodes, and plot all three interpolants against f.
# Use plt.ylim(-1, 2) or the wild oscillations near the endpoints will squash the plot.
# Print the max error for each. It gets WORSE as you add nodes.

In [ ]:
# Step 1.8: The fix: Chebyshev nodes.
# The problem in 1.7 is the equal spacing, not the polynomial degree.
# Chebyshev nodes cluster toward the endpoints:
#     x_k = cos((2k + 1) * pi / (2n + 2))    for k = 0, 1, ..., n     on [-1, 1]
# Redo 1.7 with these nodes at the same three counts, plot, and print the max errors.
# The error should now DECREASE as you add nodes.
# Write a sentence in a comment on what this means for choosing where to sample data.

---
# Exercise 2: Real functions vs a 2nd-degree Taylor approximation

## A different kind of approximation

Interpolation uses **values at many points**. Taylor uses **derivatives at one point**.
The second-degree Taylor polynomial of $f$ about $a$ is

$$p_2(x) = f(a) + f'(a)(x-a) + \frac{f''(a)}{2}(x-a)^2$$

and by Taylor's theorem the error is exactly

$$R_2(x) = \frac{f^{(3)}(\xi)}{6}(x-a)^3 \qquad \text{for some } \xi \text{ between } a \text{ and } x$$

## What that error formula is telling you

Everything hinges on the $(x-a)^3$:

- **Near $a$ it is superb.** Halving the distance from $a$ divides the error by **8**.
- **Away from $a$ it is hopeless.** The error grows cubically, and there is nothing
  holding the approximation to the function anywhere except at $a$.

This is the exact opposite of an interpolating polynomial, which is pinned to $f$ at
every node and spreads its error over the whole interval instead of concentrating all its
accuracy at one point.

Neither is "better". They answer different questions:

| | Taylor | Interpolation |
|---|---|---|
| needs | derivatives at one point | values at several points |
| accurate | near $a$ | across the whole interval |
| error | $O(|x-a|^3)$, unbounded | bounded, oscillating between nodes |
| use when | analysing local behaviour, deriving methods | you have data, or need a global fit |

Almost every method in the estimation notebook was *derived* from a Taylor expansion:
so this exercise is really about understanding where those methods' error terms came from.

In [ ]:
# Step 2.1: Taylor by hand, for exp.
# For f(x) = e**x about a = 0, all derivatives at 0 equal 1, so
#     p2(x) = 1 + x + x**2 / 2
# Write it as a function taylor2_exp(x), then plot f and p2 together on [-2, 2].
# Mark the point of expansion with plt.scatter or plt.axvline. Label everything.

In [ ]:
# Step 2.2: Look at the error.
# On the same interval, plot |f(x) - p2(x)|: first on ordinary axes, then with
# plt.semilogy on the y-axis.
# Print the error at x = 0.1, 0.5, 1.0 and 2.0.
# Roughly how many times bigger does the error get each time x doubles? Compare that
# with what the (x - a)**3 term predicts.

In [ ]:
# Step 2.3: Measure the order of the error.
# Take h = np.logspace(-4, 0, 50) and evaluate the error at x = h (so h is exactly the
# distance from the expansion point). Plot error against h on a log-log plot and fit
# the slope with np.polyfit(np.log(h), np.log(err), 1)[0].
# You should get very close to 3. That IS the theorem, measured.

In [ ]:
# Step 2.4: A general 2nd-degree Taylor function.
# Write  taylor2(f, fp, fpp, a, x)  taking the function and its first two derivatives.
# Test it against your hard-coded version from 2.1 for exp about 0; they must agree
# to machine precision.
# Then use it for  f = np.sin  about a = 0. Note that f''(0) = 0, so the quadratic
# term vanishes and p2(x) = x. Plot f and p2 on [-pi, pi] and comment on where the
# familiar "small angle approximation, sin x = x" stops being acceptable; say, where
# the relative error first exceeds 1%.

In [ ]:
# Step 2.5: Taylor from NUMERICAL derivatives.
# You will often lack analytic derivatives. Rebuild taylor2 using your central_diff
# and second_diff from the estimation notebook to get f'(a) and f''(a).
# Apply it to f(x) = np.log(1 + x) about a = 0 (where the true p2 is x - x**2 / 2).
# Compare the numerically-obtained coefficients with the exact ones, and check that
# your choice of h is the sensible one you identified in the estimation notebook.

In [ ]:
# Step 2.6: The head-to-head comparison. (The point of this exercise.)
# For f = np.exp on the interval [-2, 2], build BOTH:
#   (a) the 2nd-degree Taylor polynomial about a = 0
#   (b) the degree-2 interpolating polynomial through 3 equally spaced nodes
#       (-2, 0, 2): use your Newton code from Exercise 1
# Both are quadratics, so this is a fair fight.
# Plot f, (a) and (b) on one figure, and their two error curves on another.
# Then print, for each: the error at x = 0, and the MAXIMUM error over [-2, 2].
# Which wins on each measure? Write two or three sentences explaining why, in terms
# of what information each method was given.

---
# Exercise 3: Lagrange interpolating polynomials

## The form

Lagrange builds the polynomial from **cardinal basis functions**. For each node $i$,

$$L_i(x) = \prod_{\substack{j=0 \\ j \ne i}}^{n} \frac{x - x_j}{x_i - x_j}$$

Look at what this does. The numerator vanishes at every node *except* $x_i$; the
denominator is exactly the value the numerator takes at $x_i$. So

$$L_i(x_j) = \begin{cases} 1 & i = j \\ 0 & i \ne j \end{cases}$$

Each $L_i$ is a switch that is on at its own node and off at all the others. The
interpolant is then just a weighted sum:

$$p(x) = \sum_{i=0}^{n} y_i\, L_i(x)$$

At $x = x_j$ every term dies except the $j$-th, leaving $y_j$. Interpolation is immediate; which is why this form is the one used to *prove* things.

## Newton or Lagrange?

Same polynomial, different trade-offs:

| | Newton | Lagrange |
|---|---|---|
| coefficients | divided-difference table, $O(n^2)$ once | none needed |
| evaluating at one $x$ | $O(n)$ via Horner | $O(n^2)$ |
| adding a node | one new term, old work reused | **everything** recomputed |
| best for | computation, adaptive refinement | proofs, error analysis, deriving quadrature rules |

The error formula is stated in this basis too, and it should look familiar after
Exercise 2:

$$f(x) - p(x) = \frac{f^{(n+1)}(\xi)}{(n+1)!}\prod_{i=0}^{n}(x - x_i)$$

That product term is what causes Runge's phenomenon: with equally spaced nodes it becomes
enormous near the ends of the interval, and Chebyshev nodes are precisely the choice that
makes it as small as possible.

In [ ]:
# Step 3.1: One basis polynomial.
# Write  lagrange_basis(i, nodes, x)  returning L_i evaluated at x (scalar or array).
# Hint: loop over j, skipping j == i, multiplying (x - nodes[j]) / (nodes[i] - nodes[j])
# into a running product that starts at 1 (or np.ones_like(x)).

In [ ]:
# Step 3.2: Test the cardinal property. (Do this before anything else.)
nodes = np.array([0.0, 1.0, 2.0, 3.0])
# Build the matrix  L[i, j] = lagrange_basis(i, nodes, nodes[j])  and print it.
# It MUST be the identity matrix. Check with np.allclose(L, np.eye(len(nodes))).
# This one test catches essentially every implementation error possible here.

In [ ]:
# Step 3.3: See what the basis functions look like.
# Plot all four L_i over [-0.5, 3.5] on one figure, with a legend, plus a horizontal
# line at y = 0 and y = 1 and the nodes marked.
# Confirm visually that each curve is 1 at its own node and 0 at the others.
# Note how large they get outside the node range; that is extrapolation blowing up.

In [ ]:
# Step 3.4: The interpolant.
# Write  lagrange_interp(nodes, values, x) = sum_i values[i] * L_i(x) .
# Test it: it must reproduce the values at the nodes (np.allclose), and it must
# reproduce a degree-3 polynomial exactly across a whole interval, as in step 1.4.

In [ ]:
# Step 3.5: Uniqueness, demonstrated.
# Take f = np.cos and 6 nodes on [0, 3].
# Evaluate BOTH your Newton interpolant and your Lagrange interpolant on 500 points
# across the interval and compare them with np.allclose.
# Print the maximum difference between the two. It should be at round-off level:
# two different algorithms, one polynomial.

In [ ]:
# Step 3.6: The cost difference, measured.
# Use %timeit to compare, for 15 nodes and 1000 evaluation points:
#   (a) newton_coefficients + newton_eval
#   (b) lagrange_interp
# Then time what happens when you ADD one node to each: Newton needs one more
# coefficient, Lagrange needs the whole thing again.
# Report the numbers and say which form you would use for adaptive refinement.

In [ ]:
# Step 3.7: Interpolating real data.
# A thermocouple was read at these times:
t     = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5])
temp  = np.array([20.0, 24.8, 31.2, 39.1, 48.5, 59.4])
# 1. Interpolate and plot the curve through the data, with the points marked.
# 2. Estimate the temperature at t = 1.25 and t = 2.25.
# 3. Now estimate it at t = 4.0, outside the data range. Plot the curve out to t = 5.
#    Would you report that number to anyone? Write a sentence on interpolation versus
#    EXTRAPOLATION, and connect it to what you saw in step 3.3.

In [ ]:
# Step 3.8: Stretch: the error formula, checked numerically.
# For f = np.exp with 4 equally spaced nodes on [0, 2], the theoretical error is
#     f(x) - p(x) = f4(xi) / 4! * (x - x0)(x - x1)(x - x2)(x - x3)
# Since the 4th derivative f4 = exp, the factor f4(xi) lies between exp(0) and exp(2) for xi in [0,2].
# Plot the ACTUAL error against the two bounds you get from those extreme values
# (both times the node product / 24). The actual error should sit between them.

---
## What to take away

- **The interpolating polynomial is unique.** Newton and Lagrange are two ways of writing
  it. Choose by cost and by what you need to do next, never expecting different answers.
- **Newton's form is the computational one**: divided differences once, Horner to
  evaluate, and adding a node is nearly free.
- **Lagrange's form is the theoretical one**: the cardinal property makes interpolation
  obvious and gives you the error formula, which is where quadrature rules and finite
  difference formulas come from.
- **More nodes is not automatically better.** Runge's phenomenon is what happens when you
  push degree up with equally spaced nodes; the node *placement* matters more than the
  count.
- **Taylor and interpolation trade the same total error differently.** Taylor concentrates
  its accuracy at one point; interpolation spreads it over an interval. Know which you
  actually need before choosing.

Next: the **optimization** notebook, where the derivative estimates from notebook 1 start
driving a search rather than describing a curve.